# A comparison of holography X-ray phase contrast imaging:

# mean intensity vs intensity correlations

X-ray holography with intensity correlations is given by the following forward problem:
$$F(f)=c_{f} ~~\text{with}~~ c_{f}(x,y):=\operatorname{Cov}(I(x),I(y)),~~\text{and}~~I:=|\mathcal{D}e^{f}u|^2 $$
The forward operator is given by:
$$
F(f):=|\mathcal{D}M_{\exp(f)}\operatorname{Cov}[u]M_{\exp(f)}^*\mathcal{D}^*|^2.
$$
In contrast to the imaging from intensity correlations, the X-ray holography with mean intensity is given by the following forward problem:
$$G(f)=d_{f} ~~\text{with}~~d_{f}(x):=c_{f}(x,x)=\mathbb{E}[I(x)]^2$$
The forward operator is given by:
$$
G(f):=\operatorname{Diag}(\mathcal{D}M_{\exp(f)}\operatorname{Cov}[u]M_{\exp(f)}^*\mathcal{D}^*).
$$

In [ ]:
import os
import sys

#sys.path.append(os.path.join(os.path.dirname(__file__), '../../'))

from regpy.hilbert import L2
from regpy.vecsps import UniformGridFcts
from regpy.vecsps import NumPyVectorSpace
from regpy.solvers import Setting
from regpy.solvers.nonlinear.fista import FISTA
import regpy.stoprules as rules
from regpy.operators import SquaredModulus, VectorOfOperators, Identity, RealPart, ImaginaryPart
from regpy.functionals import QuadraticNonneg, QuadraticBilateralConstraints
import matplotlib.pyplot as plt
#from regpy.operators.fresnel import fresnel_propagator

from auxiliary_ops import ReIm, summation, fresnel_prop,  Proj, Reshape
from phaseless_passive_ip_ops import Mat,  Tau 
from low_rank_op_misfit_fct import HilbertSchmidtLowRank
from create_Vcov import _create_Vcov

import numpy as np
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)-20s :: %(message)s'
)


Setting up parameters 

In [ ]:
N=256                             # Pixel numbers
N_frame=3000                      # Number of frames ( or realizations)
T=1e9                             # the observation time or the number of photon counts
fresnel_number=10                 # not properly scaled
N_b=4                             # denotes the rank of the matrix V
sigma=0.5                         # parameter used in the rapid deacaying function
regpar_intcorr=1e-10              # regularization parameter for intensity correlations
Nfista_intcorr=200                # number of FISTA iterations for intensity correlations
Nfista_meanint=200                # number of FISTA iterations for  mean intensity
regpar_meanint=1e-10              # regularization parameter for mean intensity
test_image = 2                    # choose different test image 1 or 2

Perform a uniform grid

In [ ]:
xsample=np.arange(-1,1-1/N,2/N)
ysample=xsample
grid=UniformGridFcts((-1, 1, N), (-1, 1, N), dtype=complex)
grid_domain=UniformGridFcts((-1, 1, 2), (-1, 1, N), (-1, 1, N))

fp=fresnel_prop(grid, number=complex(0, 1)/(2*fresnel_number))   # perform Fresenel propagation operator


construct the decomposition $\operatorname{Cov}[u]=VV^*$ of the covariance operator 

In [ ]:
create_types=['spatial', 'Fresnelprop', 'fourier_random']
create_type='fourier_random'

Vcov, S=_create_Vcov(N, N_b, create_type=create_type, grid=grid, sigma=sigma, xsample=xsample, ysample=ysample)
# here S is the singular values of the matrix V

Discretization of the spaces

In [ ]:
grid_codomain= NumPyVectorSpace((N, N, N_b), dtype=complex)
grid_codomain_2=NumPyVectorSpace((N, N, N_b, N_b), dtype=complex)
grid_codomain_3=NumPyVectorSpace((2, N, N, N_b, N_b), dtype=complex)
grid_codomain_4=NumPyVectorSpace((N_frame, N, N))
grid_codomain_5=NumPyVectorSpace((N, N, N_b**2), dtype=complex)
grid_dom=UniformGridFcts(np.arange(2), xsample, ysample)
grid_codomain_6=NumPyVectorSpace((N, N))

# Compute the forward operator associated with the intensity correlations

In [ ]:
Mat_op=Mat(grid, grid_codomain, Vcov, fp)
Tau_op=Tau(NumPyVectorSpace(shape=(N_b,),dtype=complex), NumPyVectorSpace(shape=(N,N),dtype=complex))
Proj_op=Proj(grid_codomain_2, grid_codomain_3)
#Theta_op=Theta_2(grid_codomain_3, grid_codomain_4, N, N_b) #The codomain has the dimension of the intensities
Resh=Reshape(grid_codomain_2, grid_codomain_5)
ReIm_op=ReIm(grid, grid_domain)
op_intcorr=Resh*Tau_op*Mat_op

# Compute the forward operator associated with the mean intensity

In [ ]:
power=SquaredModulus(grid_codomain)
trace=summation(power.codomain, grid_codomain_6)
op_meanint=trace*power*Mat_op

# Test image

In [ ]:
if test_image == 1:
    X,Y = np.meshgrid(xsample, ysample, sparse=False)
    absorp_0=(abs(X)<0.601)*(abs(Y)<0.199)+(abs(X)<0.199)*(abs(Y)<0.601)
    absorp_0=absorp_0.astype('int')
    absorp_1=(X**2+Y**2<=0.501**2)*(X**2+Y**2>=0.45**2)
    absorp_1=absorp_1.astype('int')
    absorp_2=(abs(X)<0.3)*(abs(Y)<0.055)+(abs(X)<0.055)*(abs(Y)<0.3)
    absorp_2=absorp_2.astype('int')
    absorp_3=(abs(X)**25+abs(Y)**25<=0.601**25)*(abs(X)**25+abs(Y)**25>=0.551**25)
    absorp_3=absorp_3.astype('int')
    absorp=absorp_0+absorp_1+absorp_2+absorp_3
    phase = ((abs(X+Y) <=0.101)+(abs(X-Y) <= 0.101)).astype('int')
    support_mask=((abs(X)**2+abs(Y)**2)<=0.7).astype('int')   # constant circular bump
    contrast = support_mask*(0.1*absorp+0.1*complex(0,1)*phase) 
    mask=(contrast!=0)                                         # defines the support of the contrast
elif test_image == 2:
    X,Y = np.meshgrid(xsample, ysample, sparse=False)
    absorp=np.load('cell1.npy')
    phase=np.load('cell2.npy')
    #absorp=np.load('cell256.npy')
    #phase=absorp
    #support_mask=((abs(X)<=0.801)*(abs(Y)<=0.801)).astype('int')              #constant rectangular bump
    support_mask=((abs(X)**2+abs(Y)**2)<=0.6).astype('int')                    # constant circular bump
    contrast = (0.01+0.01*complex(0,1))*support_mask+(0.2*absorp + 0.2*complex(0,1) * phase)
else:
    raise ValueError

Create noisy intensity data and performing time-resolved

In [ ]:
ptw_detection= SquaredModulus(grid)
taumat_0=Mat_op(contrast) 
intens_tot=np.zeros((N, N))
intensities=np.zeros((N_frame, N, N))

for i in range(0, N_frame):
    print(i)
    random=1/np.sqrt(2)*(np.random.randn(N_b)+complex(0,1)*np.random.randn(N_b))
    uinc=np.tensordot(taumat_0, random, axes=([-1], [0]))
    signal=ptw_detection(uinc)
    
    #Cox-processes
    signal=(1/T)*np.random.poisson(lam=T*signal.flatten(), size=(N**2)).reshape(N, N)
    intens_tot+=signal
    intensities[i, :, :]=signal
    

intensities-=intens_tot/N_frame

data_intcorr=intensities.transpose([1, 2, 0])/np.sqrt(N_frame)  # intensity data
data_meanint=intens_tot/N_frame                                 # mean intensity


# Inversion FISTA for intensity correlations

In [ ]:
dom = op_intcorr.codomain
Sfun = HilbertSchmidtLowRank(domain = dom+dom, data=[data_intcorr])

taumat=op_intcorr(contrast)
taumat_join=Sfun.domain.join(taumat, taumat)
Sfun.getLipschitz(taumat_join)

Double = VectorOfOperators([Identity(op_intcorr.codomain), Identity(op_intcorr.codomain)])

Re=RealPart(grid)

Im=ImaginaryPart(op_intcorr.domain)

penalty=QuadraticNonneg(ReIm_op.codomain)

setting = Setting(op=Double*op_intcorr*ReIm_op.adjoint, penalty=penalty, data_fid = Sfun,regpar=regpar_intcorr)

FISTA_solver = FISTA(setting)

stoprule = (rules.CountIterations(Nfista_intcorr))

reco_intcorr, reco_data_intcorr=FISTA_solver.run(stoprule)
reco_intcorr=ReIm_op.adjoint(reco_intcorr)


# Inversion FISTA for mean intensity

In [ ]:
ub=ReIm_op.codomain.zeros()
ub[0]=0.1
ub[1]=0.2

data_space = L2(op_meanint.codomain)

penalty=QuadraticNonneg(ReIm_op.codomain)

penalty_2= QuadraticBilateralConstraints(ReIm_op.codomain, lb=0, ub=ub)

setting = Setting(op=op_meanint*ReIm_op.adjoint, penalty=penalty, data_fid = data_space, data_fid_shift=data_meanint,regpar=regpar_meanint)

FISTA_solver = FISTA(setting)

stoprule = (rules.CountIterations(Nfista_meanint))

reco_meanint, reco_data_meanint=FISTA_solver.run(stoprule)

reco_meanint=ReIm_op.adjoint(reco_meanint)

# Relative error of the two problems

In [ ]:
error_meanint=np.linalg.norm(reco_meanint-contrast)/np.linalg.norm(contrast)
error_intcorr=np.linalg.norm(reco_intcorr-contrast)/np.linalg.norm(contrast)
print(f"error_meanint = {error_meanint}")
print(f"error_intcorr = {error_intcorr}")

# Plots

In [ ]:
vmin=0
vmax=0.2
#cmap='Reds'
fontsize=10
levels=40
fig, axs = plt.subplots(2, 3, figsize=(14, 7))
# Plot each image

im1 = axs[0, 0].imshow(contrast.real, vmin=vmin, vmax=vmax)
axs[0, 0].set_title('Exact absorption',fontsize=fontsize)
axs[0, 0].axis('off')
fig.colorbar(im1, ax=axs[0, 0])

im2 = axs[0, 1].imshow(reco_intcorr.real, vmin=vmin, vmax=vmax)
axs[0, 1].set_title('Recovered absorption-IntCorr',fontsize=fontsize)
axs[0, 1].axis('off')
fig.colorbar(im2, ax=axs[0, 1])

im3 = axs[1, 0].imshow(contrast.imag, vmin=vmin, vmax=vmax)
axs[1, 0].set_title('Exact phase',fontsize=fontsize)
axs[1, 0].axis('off')
fig.colorbar(im3, ax=axs[1, 0])

im4 = axs[1, 1].imshow(reco_intcorr.imag, vmin=vmin, vmax=vmax)
axs[1, 1].set_title('Recovered phase-IntCorr',fontsize=fontsize)
axs[1, 1].axis('off')
fig.colorbar(im4, ax=axs[1, 1])

im5 = axs[0, 2].imshow(reco_meanint.real, vmin=vmin, vmax=vmax)
axs[0, 2].set_title('Recovered absorption-MeanInt',fontsize=fontsize)
axs[0, 2].axis('off')
fig.colorbar(im5, ax=axs[0, 2])

im6 = axs[1, 2].imshow(reco_meanint.imag, vmin=vmin, vmax=vmax)
axs[1, 2].set_title('Recovered phase-MeanInt',fontsize=fontsize)
axs[1, 2].axis('off')
fig.colorbar(im6, ax=axs[1, 2])

# Adjust layou
plt.tight_layout()
#plt.subplot_tool()
plt.show()


plt.figure(figsize=(14, 7))
plt.plot(reco_meanint[int(N/2), :].imag, label='Mean intensity')
plt.plot(reco_intcorr[int(N/2), :].imag, label='Intensity correlations')
plt.plot(contrast[int(N/2), :].imag, label='Exact phase')
plt.legend()
plt.show()

plt.figure(figsize=(14, 7))
plt.plot(reco_meanint[int(N/2), :].real, label='Mean intensity')
plt.plot(reco_intcorr[int(N/2), :].real, label='Intensity correlations')
plt.plot(contrast[int(N/2), :].real, label='Exact absorption')
plt.legend()
plt.show()

